# Box-Jenkins Time Series Modeling Workflow
### Applied Time Series Analysis (ATSA) Educational Notebook

This tutorial demonstrates the three classic stages of the Box-Jenkins methodology:
1. **Identification**: Stationarity verification (ADF & KPSS tests) and ACF/PACF analysis.
2. **Estimation**: Auto-ARIMA parameter selection minimizing AIC.
3. **Diagnostic Checking**: Ljung-Box test for residual white noise and Jarque-Bera normality.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera

# 1. Generate Synthetic Airline Passenger Style Non-Stationary Series
np.random.seed(42)
t = np.arange(120)
trend = 0.5 * t
seasonality = 10 * np.sin(2 * np.pi * t / 12)
noise = np.random.normal(0, 2, 120)
series = pd.Series(50 + trend + seasonality + noise, index=pd.date_range('2014-01-01', periods=120, freq='MS'))

plt.figure(figsize=(10, 4))
plt.plot(series, label='Original Series')
plt.title('Non-Stationary Time Series with Trend & Seasonality')
plt.legend()
plt.show()

## Stage 1: Identification & Differencing
We test stationarity using the Augmented Dickey-Fuller (ADF) test.

In [ ]:
adf_raw = adfuller(series.dropna())
print(f"Raw Series ADF p-value: {adf_raw[1]:.4f} (Stationary: {adf_raw[1] <= 0.05})")

# Apply first order differencing d=1
diff_series = series.diff().dropna()
adf_diff = adfuller(diff_series)
print(f"First Difference ADF p-value: {adf_diff[1]:.4f} (Stationary: {adf_diff[1] <= 0.05})")

## Stage 2: Estimation (Fitting ARIMA)
We fit candidate ARIMA models and select the one minimizing the Akaike Information Criterion (AIC).

In [ ]:
best_aic = float('inf')
best_order = None
for p in range(3):
    for q in range(3):
        try:
            model = ARIMA(series, order=(p, 1, q))
            res = model.fit()
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = (p, 1, q)
        except Exception:
            continue

print(f"Optimal Model: ARIMA{best_order} with AIC: {best_aic:.2f}")
fitted_model = ARIMA(series, order=best_order).fit()

## Stage 3: Diagnostic Checking
Check whether residuals resemble white noise.

In [ ]:
residuals = fitted_model.resid[1:]
lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
jb_stat, jb_pvalue = jarque_bera(residuals)

print(f"Ljung-Box p-value: {lb_test['lb_pvalue'].iloc[0]:.4f} (White Noise: {lb_test['lb_pvalue'].iloc[0] > 0.05})")
print(f"Jarque-Bera p-value: {jb_pvalue:.4f} (Normal Distribution: {jb_pvalue > 0.05})")